# 📂 파일 입출력

**주제**: 텍스트 파일과 CSV 파일을 읽고 쓰기

| 섹션 | 내용 |
|------|------|
| 1. `open` 의 기본 | 모드, 쓰기/읽기 |
| 2. `with` 구문 | 자동으로 파일 닫기 (권장) |
| 3. CSV 파일 다루기 | `csv` 모듈 |
| 4. URL에서 직접 읽기 | `urllib.request` |
| 5. 인코딩 문제 | `utf-8`, `cp949` |


---
## 1. `open` 의 기본

```python
f = open(파일경로, 모드, encoding=...)
... 작업 ...
f.close()
```

| 모드 | 의미 |
|------|------|
| `'r'` | 읽기 (기본) |
| `'w'` | 쓰기 — **기존 내용을 덮어쓴다** |
| `'a'` | 추가 (append) |
| `'rb'`, `'wb'` | 바이너리 모드 |

In [ ]:
# 쓰기
f = open('text.txt', 'w', encoding='utf-8')
f.write('첫 줄입니다.\n')
f.write('두 번째 줄.\n')
f.close()

# 읽기
f = open('text.txt', 'r', encoding='utf-8')
content = f.read()
print(content)
f.close()

---
## 2. `with` 구문 — 자동으로 닫기 (권장)

`with` 블록을 빠져나가면 파일이 자동으로 닫힌다. 예외가 발생해도 안전.

In [ ]:
# 쓰기
with open('text.txt', 'w', encoding='utf-8') as f:
    f.write('Hello\nWorld\nPython\n')

# 읽기 — 한 번에
with open('text.txt', 'r', encoding='utf-8') as f:
    text = f.read()
print(text)

### 2.1 줄 단위로 읽기

In [ ]:
with open('text.txt', 'r', encoding='utf-8') as f:
    for line in f:
        print(line.rstrip())   # 줄바꿈 문자 제거

# 전부 리스트로 받기
with open('text.txt', 'r', encoding='utf-8') as f:
    lines = f.readlines()
print(lines)

---
## 3. CSV 파일 다루기

CSV(Comma-Separated Values)는 표 데이터를 평문으로 저장하는 가장 보편적인 포맷.
파이썬은 표준 `csv` 모듈을 제공한다.

In [ ]:
import csv

# 쓰기
rows = [
    ['name', 'age', 'city'],
    ['Alice', 30, 'Seoul'],
    ['Bob',   25, 'Busan'],
    ['Carol', 28, 'Incheon'],
]
with open('people.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(rows)

# 읽기
with open('people.csv', 'r', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)        # 첫 줄(헤더) 따로
    print('헤더:', header)
    for row in reader:
        print(row)

### 3.1 딕셔너리로 읽기 — `DictReader`

In [ ]:
import csv

with open('people.csv', 'r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row['name'], '/', row['age'], '/', row['city'])

---
## 4. URL에서 직접 읽기

다운로드 없이 웹의 CSV를 바로 읽을 수도 있다.

In [ ]:
import csv, urllib.request, io

url = 'https://raw.githubusercontent.com/dongupak/DataSciPy/master/data/csv/weather.csv'

try:
    with urllib.request.urlopen(url, timeout=10) as resp:
        raw = resp.read().decode('cp949')   # 한글 파일 — 인코딩 주의
    reader = csv.reader(io.StringIO(raw))
    header = next(reader)
    print('헤더:', header)
    for i, row in enumerate(reader):
        if i < 3:
            print(row)
        else:
            break
except Exception as e:
    print('네트워크 오류:', e)

---
## 5. 인코딩 문제

한글 파일은 **저장한 인코딩** 으로 열어야 깨지지 않는다.

| 인코딩 | 사용처 |
|--------|--------|
| `utf-8` | 가장 일반적, 권장 |
| `cp949` (≈ `euc-kr`) | 윈도우에서 만든 한글 파일 |
| `latin-1` | 깨지지 않게 1바이트씩 읽기 (임시 방편) |

```python
open('file.csv', encoding='utf-8')   # 안 되면
open('file.csv', encoding='cp949')
```

---
## ✏️ 연습문제

**Q.** `people.csv` 를 읽어 각 사람의 나이를 1살씩 더한 새 파일 `people_next_year.csv` 를 만들어라.

In [ ]:
import csv

with open('people.csv', 'r', encoding='utf-8') as fin, \
     open('people_next_year.csv', 'w', newline='', encoding='utf-8') as fout:
    reader = csv.DictReader(fin)
    writer = csv.DictWriter(fout, fieldnames=reader.fieldnames)
    writer.writeheader()
    for row in reader:
        row['age'] = int(row['age']) + 1
        writer.writerow(row)

# 확인
with open('people_next_year.csv', encoding='utf-8') as f:
    print(f.read())